In [0]:
%sql
LIST '/Volumes/workspace/dbacademy/events-kafka'

In [0]:
%sql
SELECT *
FROM text.`/Volumes/workspace/dbacademy/events-kafka`
LIMIT 5;

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/dbacademy/events-kafka',
    format => 'json'
) LIMIT 5;

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.kafka_event_bronze_raw;

CREATE TABLE dbacademy.kafka_event_bronze_raw AS
SELECT *
FROM read_files(
    '/Volumes/workspace/dbacademy/events-kafka',
    format => 'json'
) ;

SELECT * FROM dbacademy.kafka_event_bronze_raw LIMIT 5;

In [0]:
%sql
CREATE OR REPLACE TABLE dbacademy.kafka_event_bronze_decoded AS
SELECT 
    cast(unbase64(key) as string) AS decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    cast(unbase64(value) as string) AS decoded_value
FROM dbacademy.kafka_event_bronze_raw;

SELECT * FROM dbacademy.kafka_event_bronze_decoded LIMIT 5;

# Working with JSON formatted strings in a table
## 1 - Flattening JSON String columns

In [0]:
%sql
SELECT  decoded_value:device,
        decoded_value:traffic_source,
        decoded_value:geo,   --contains another JSON formatted string
        decoded_value:items, --contains a nested-array of JSON formatted string 
        decoded_value
FROM dbacademy.kafka_event_bronze_decoded
LIMIT 5;

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.kafka_event_bronze_string_flattened;

CREATE TABLE dbacademy.kafka_event_bronze_string_flattened AS
SELECT  decoded_key,
        offset,
        partition,
        timestamp,
        topic,
        decoded_value:device,
        decoded_value:traffic_source,
        decoded_value:geo,   --contains another JSON formatted string
        decoded_value:items --contains a nested-array of JSON formatted string 
FROM dbacademy.kafka_event_bronze_decoded;

SELECT * FROM dbacademy.kafka_event_bronze_string_flattened LIMIT 5;

## 2 - Flattening JSON formatted String via Struct conversion

In [0]:
%sql
SELECT schema_of_json('{"device":"Android","ecommerce":{},"event_name":"main","event_timestamp":1593880885036129,"geo":{"city":"New York","state":"NY"},"items":[],"traffic_source":"google","user_first_touch_timestamp":1593880885036129,"user_id":"UA000000107398054"}') as schema

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.kafka_event_bronze_struct;

CREATE TABLE dbacademy.kafka_event_bronze_struct AS
SELECT  * EXCEPT(decoded_value),
        from_json(decoded_value, schema_of_json('{"device":"Android","ecommerce":{},"event_name":"main","event_timestamp":1593880885036129,"geo":{"city":"New York","state":"NY"},"items":[],"traffic_source":"google","user_first_touch_timestamp":1593880885036129,"user_id":"UA000000107398054"}')) as value
FROM dbacademy.kafka_event_bronze_decoded;

SELECT * FROM dbacademy.kafka_event_bronze_struct LIMIT 5;


### 2.1 - Extract fields, nested fields, and nested array from STRUCT column

In [0]:
%sql
SELECT decoded_key,
        value.device,   --field
        value.geo.city, --nested field
        value.items as items, --nested array
        array_size(items) as no_elements_in_array --number of elements in the array
FROM dbacademy.kafka_event_bronze_struct
ORDER BY no_elements_in_array DESC

### 2.2 Explode array


In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.kafka_event_bronze_explode_array;

CREATE TABLE dbacademy.kafka_event_bronze_explode_array AS
SELECT decoded_key,
    array_size(value.items) as no_elements_in_array,
    explode(value.items) as item_in_array,
    value.items as items
FROM dbacademy.kafka_event_bronze_struct;


SELECT * FROM dbacademy.kafka_event_bronze_explode_array
ORDER BY no_elements_in_array DESC, decoded_key


## 3 - Working with VARIANT column

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.kafka_event_bronze_variant;

CREATE TABLE dbacademy.kafka_event_bronze_variant AS
SELECT decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    parse_json(decoded_value) as json_variant
FROM dbacademy.kafka_event_bronze_decoded;

In [0]:
%sql
SELECT decoded_key,
        json_variant,
        json_variant:device :: string as device,
        json_variant:items
FROM dbacademy.kafka_event_bronze_variant